In [1]:
import pandas as pd
import numpy as np
import talib
from math import atan

# 设定股票池

In [2]:
stocks = get_index_stocks('000300.XSHG',date='2023-01-01')
stocks2 = get_index_stocks('000300.XSHG')
stocks3 = list(set(stocks)&set(stocks2))

In [3]:
# 辅助函数
#计算近N日递增天数
def count_diff(df,n):
    
    diff = df.diff()
    diff[diff > 0] = 1
    diff[diff <= 0] = 0
    count = diff.rolling(window=n).sum()
    
    return count


In [39]:
# 计算因子函数
def process_(df):
    data  = df.copy()
    data['ma5'] = talib.MA(data['close'],timeperiod=5)
    data['ma10'] = talib.MA(data['close'],timeperiod=10)
    data['ma21'] = talib.MA(data['close'],timeperiod=21)
    data['ma55'] = talib.MA(data['close'],timeperiod=55)
    data['vol21'] = talib.MA(data['volume'],timeperiod=21)
    
    data['target2'] = data['close'].shift(-2)/data['close']  # 标签
    
    # 防止分母为零，所以加个无穷小 最后用反三角函数处理将0到无穷大 映射到0到二分之pi 
    data['factor0'] = data.apply(lambda x: atan((x['high']-x['close'])/(x['close']-x['low']+0.00000001)),axis=1)
    data['factor1'] = data.apply(lambda x: atan((x['high']-x['open'])/(x['close']-x['low']+0.00000001)),axis=1)
    data['factor2'] = data.apply(lambda x: atan((x['open']-x['low'])/(x['high']-x['close']+0.00000001)),axis=1)
    data['factor3'] = data.apply(lambda x: atan((x['close']-x['low'])/(x['high']-x['close']+0.00000001)),axis=1)
    
    data['increase5'] = count_diff(df['close'],5) # 近5日价格递增的天数
    data['increase_vol5'] = count_diff(df['volume'],5)
    
    data['return1'] = (data['close']/data['close'].shift(1)-1)*100
    data['return3'] = (data['close']/data['close'].shift(3)-1)*100
    data['return5'] = (data['close']/data['close'].shift(5)-1)*100 #5日涨幅
    
    data['bias5'] = (data['close']/data['ma5']-1)*100 #乖离率
    data['bias10'] = (data['close']/data['ma10']-1)*100
    data['bias21'] = (data['close']/data['ma21']-1)*100
    data['bias55'] = (data['close']/data['ma55']-1)*100
   
    data['bias5_'] = data['bias5']-data['bias5'].shift(1)#乖离率的1阶差分
    data['bias10_'] = data['bias10']-data['bias10'].shift(1)
    data['bias21_'] = data['bias21']-data['bias21'].shift(1)
    data['bias55_'] = data['bias55']-data['bias55'].shift(1)
    
    data['trend5'] = (data['ma5']/data['ma5'].shift(3)-1)*100 # ma5的3日变化率
    data['trend21'] = (data['ma21']/data['ma21'].shift(5)-1)*100 # ma21的5日变化率
    data['high5'] = (data['high']/data['ma5']-1)*100
    data['high10'] = (data['high']/data['ma10']-1)*100
    data['high21'] = (data['high']/data['ma21']-1)*100
    
    data['k1'] = (data['close']-data['open'])/data['close'].shift(1)*2.5 #当日阳线或阴线的长度
    data['k1_1'] = data['k1'].shift(1)
    data['k1_2'] = data['k1'].shift(2)
    data['hc'] = (data['high']/data['close']-1)*100
    data['return1_'] = data['return1']-data['return1'].shift(1)#收益率的一阶差分
    data['return3_'] = data['return3']-data['return3'].shift(1)
    data['return5_'] = data['return5']-data['return5'].shift(1)
    
    data['vol21_'] = (data['volume']/data['vol21']) 
    data['vol21_1'] = data['vol21_']-data['vol21_'].shift(1)
    data['vol21_2'] = data['vol21_']-data['vol21_'].shift(2)
    data['vol21_3'] = data['vol21_']-data['vol21_'].shift(3)
    
    return data.dropna()

# 合并20年-22年 训练数据

In [5]:
target_col = [ 'target2','factor0','factor1','factor2','factor3','return1', 'return3', 'return5', 'bias5', 'bias10', 'bias21', 'bias55',
        'bias5_', 'bias10_', 'bias21_', 'bias55_', 'trend5','increase5','increase_vol5',
       'trend21', 'high5', 'high10', 'high21', 'k1', 'k1_1', 'k1_2', 'hc',
       'return1_', 'return3_', 'return5_','vol21_', 'vol21_1', 'vol21_2',
       'vol21_3']

In [6]:
df = get_price(stocks3[0],'2020-01-01', '2023-01-01','1d')
train = process_(df)[target_col]  #计算因子

for x in stocks3[1:]:
    df = get_price(x,'2020-01-01', '2023-01-01','1d')
    df2= process_(df)[target_col]
    train = pd.concat([train,df2],axis=0) #拼接数据

In [7]:
# 因子范围
train.describe().T

,count,mean,std,min,25%,50%,75%,max
target2,189127.0,1.001788,0.040041,0.662128,0.980138,0.999370,1.020250,1.427299
factor0,189127.0,0.816961,0.504814,0.000000,0.340097,0.834654,1.293289,1.570796
factor1,189127.0,0.782773,0.399564,0.000000,0.534955,0.789398,1.051650,1.570796
factor2,189127.0,0.740043,0.388502,0.000000,0.471946,0.774546,0.982793,1.570796
factor3,189127.0,0.751377,0.504654,0.000000,0.274167,0.732815,1.227772,1.570796
return1,189127.0,0.088289,2.798680,-19.997773,-1.379938,0.000000,1.320427,20.010935
return3,189127.0,0.270000,4.942851,-30.845070,-2.449728,-0.054201,2.561327,43.753983
return5,189127.0,0.452668,6.456649,-36.076161,-3.166350,-0.053248,3.430365,322.171946
bias5,189127.0,0.116083,3.082903,-20.653695,-1.545730,-0.071911,1.573501,26.927406
bias10,189127.0,0.271499,4.722800,-26.993553,-2.379862,-0.073686,2.541037,91.080846


In [8]:
# 选择分布在0.1%到99.8%的数据 
df2 = train[(train<=train.quantile(q=0.999)).all(axis=1)]
df2 = df2[(df2>= train.quantile(q=0.001)).all(axis=1)]

# 保存因子边界
train.quantile(q=0.999).to_csv('top.csv',header=None)
train.quantile(q=0.001).to_csv('bottom.csv',header=None)


In [9]:
df2.describe().T

,count,mean,std,min,25%,50%,75%,max
target2,184333.0,1.001644,0.038490,0.851397,0.980360,0.999309,1.019852,1.209237
factor0,184333.0,0.817771,0.502027,0.000000,0.344647,0.835476,1.292497,1.570796
factor1,184333.0,0.782314,0.398300,0.000000,0.534744,0.789964,1.051650,1.570796
factor2,184333.0,0.738985,0.386334,0.000000,0.471778,0.774421,0.982793,1.570796
factor3,184333.0,0.750920,0.501887,0.000000,0.278300,0.732815,1.222908,1.570796
return1,184333.0,0.078156,2.603987,-10.006598,-1.352265,0.000000,1.286449,11.233752
return3,184333.0,0.227349,4.588835,-17.397260,-2.404372,-0.064725,2.478667,25.338094
return5,184333.0,0.373782,5.932122,-21.173763,-3.122772,-0.082610,3.303303,33.135392
bias5,184333.0,0.095818,2.858415,-11.830743,-1.515152,-0.075131,1.527082,14.891519
bias10,184333.0,0.218225,4.369233,-16.274189,-2.342298,-0.088914,2.451168,23.796880


# 随机森林

In [10]:
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import train_test_split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(df2.drop(columns='target2'), df2['target2'], test_size=0.1, random_state=36)

In [12]:
X_train.head()

,factor0,factor1,factor2,factor3,return1,return3,return5,bias5,bias10,bias21,bias55,bias5_,bias10_,bias21_,bias55_,trend5,increase5,increase_vol5,trend21,high5,high10,high21,k1,k1_1,k1_2,hc,return1_,return3_,return5_,vol21_,vol21_1,vol21_2,vol21_3
2021-11-19,0.262995,0.839890,0.519146,1.307802,1.396032,-3.361345,-5.866303,-1.974712,-4.570915,-5.691692,-6.840303,2.516516,1.689847,1.678264,1.161811,-3.956884,1.0,2.0,-2.058964,-1.477483,-4.086854,-5.213316,0.040411,-0.067233,-0.021008,0.507246,5.076146,3.227537,2.049800,0.727494,-1.159363,-0.084381,0.136937
2021-08-12,0.785398,0.000000,1.107148,0.785398,-0.204918,0.205761,1.247401,0.247015,0.640628,-0.891559,-4.383679,-0.454594,-0.331853,-0.059355,-0.022219,0.746578,3.0,2.0,-1.035773,0.452861,0.847282,-0.688051,-0.005123,0.010267,0.010288,0.205339,-0.410257,-1.249540,-0.207900,0.667456,-0.423877,-0.389300,-0.527417
2020-07-20,0.165149,0.785398,0.785398,1.405647,1.886792,-0.307692,-2.994012,0.371747,-1.669196,-0.366086,-1.197605,2.465343,2.345293,1.816425,1.947816,-2.595051,1.0,3.0,0.043950,0.681537,-1.365706,-0.058574,0.039308,-0.015674,-0.053846,0.308642,2.200272,2.741088,0.933480,0.816870,0.120976,-0.359655,-0.258409
2021-01-27,0.812419,0.923274,0.607185,0.758378,-1.221896,-7.314836,-3.853473,-4.970142,-4.108939,-5.800752,14.964344,-0.448232,-0.003202,-1.556316,-2.277000,0.481928,3.0,3.0,3.758420,-2.736634,-1.855191,-3.586767,0.014663,-0.201232,0.047581,2.350322,8.784701,-4.166907,-3.853473,0.649129,-0.486860,-0.676765,-0.479239
2021-03-11,1.249046,0.605545,0.834140,0.321751,-7.172131,-2.685285,-7.172131,-3.903267,-7.418761,-12.091669,-9.634949,-5.909956,-5.905139,-6.088459,-7.031302,-4.050478,1.0,2.0,-2.103311,4.369962,0.551809,-4.523402,-0.153689,0.078041,0.021482,8.609272,-12.231335,-3.096807,-2.484631,2.649006,1.764308,1.726317,1.404366


In [13]:
model = RandomForestRegressor(n_estimators=100,max_depth=7, random_state=0)
model.fit(X_train, y_train)


RandomForestRegressor(bootstrap=True, criterion='mse', max_depth=7,
           max_features='auto', max_leaf_nodes=None,
           min_impurity_decrease=0.0, min_impurity_split=None,
           min_samples_leaf=1, min_samples_split=2,
           min_weight_fraction_leaf=0.0, n_estimators=100, n_jobs=1,
           oob_score=False, random_state=0, verbose=0, warm_start=False)

In [14]:
from sklearn.metrics import r2_score, mean_squared_error

In [15]:
# 测试并计算 R2 分数以及均方误差
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print('R2 score:', r2)
print('Mean Squared Error:', mse)

R2 score: 0.008128496811839914
Mean Squared Error: 0.001480745773393013


In [16]:
import joblib

# 保存模型 
joblib.dump(model, 'model.joblib')

['model.joblib']

# 预测

预测23年以后的选股并排序

In [18]:
target_col = ['factor0','factor1','factor2','factor3','return1', 'return3', 'return5', 'bias5', 'bias10', 'bias21', 'bias55',
        'bias5_', 'bias10_', 'bias21_', 'bias55_', 'trend5','increase5','increase_vol5',
       'trend21', 'high5', 'high10', 'high21', 'k1', 'k1_1', 'k1_2', 'hc',
       'return1_', 'return3_', 'return5_','vol21_', 'vol21_1', 'vol21_2',
       'vol21_3']

In [19]:
top = pd.read_csv('top.csv',header=None,index_col=0,squeeze=True)
top = top.drop('target2')
bottom = pd.read_csv('bottom.csv',header=None,index_col=0,squeeze=True)
bottom = bottom.drop('target2')

In [29]:
# 获取预测 因子计算要用历史数据 getprice 多往前取一些
def get_predict(model,name):
    ddf = get_price(name,'2022-05-01', end_date='2023-06-21',frequency='1d')
    factor = process_(ddf) #计算因子

    factor2 = factor[target_col][(factor[target_col]<=top).all(axis=1)]
    factor2 = factor2[(factor2>=bottom).all(axis=1)] 
#     print(factor)
    tar = model.predict(factor2)
    testdf = pd.DataFrame(tar,index=factor2.index,columns=[name])
        
    return testdf,factor['target2']

In [42]:
pre_df, tar_df = get_predict(model,stocks3[0])

for x in stocks3[1:]:
    pre_df2, tar_df2 = get_predict(model,x)
    tar_df2.name = x
    pre_df = pd.concat([pre_df,pre_df2],join='outer',axis=1)
    tar_df = pd.concat([tar_df,tar_df2],join='outer',axis=1)

In [43]:
# 预测值
pre_df.head()

,601088.XSHG,000895.XSHE,603986.XSHG,002916.XSHE,600276.XSHG,002812.XSHE,688126.XSHG,601186.XSHG,000408.XSHE,000063.XSHE,600031.XSHG,601236.XSHG,600028.XSHG,600309.XSHG,600015.XSHG,300015.XSHE,600009.XSHG,600436.XSHG,300433.XSHE,600383.XSHG,600111.XSHG,300782.XSHE,000166.XSHE,601939.XSHG,601328.XSHG,688111.XSHG,600999.XSHG,000001.XSHE,300142.XSHE,300750.XSHE,601398.XSHG,688008.XSHG,600132.XSHG,002230.XSHE,688363.XSHG,601169.XSHG,600332.XSHG,600763.XSHG,601988.XSHG,600600.XSHG,...,601877.XSHG,002594.XSHE,600011.XSHG,603290.XSHG,002821.XSHE,003816.XSHE,601009.XSHG,688005.XSHG,600061.XSHG,000800.XSHE,603806.XSHG,002064.XSHE,002493.XSHE,603019.XSHG,002466.XSHE,002074.XSHE,002352.XSHE,002648.XSHE,603185.XSHG,600585.XSHG,600406.XSHG,600958.XSHG,000938.XSHE,300496.XSHE,002938.XSHE,601788.XSHG,601881.XSHG,601899.XSHG,603259.XSHG,601336.XSHG,000002.XSHE,002709.XSHE,000596.XSHE,300122.XSHE,600887.XSHG,002050.XSHE,002414.XSHE,300896.XSHE,601857.XSHG,601658.XSHG
2022-07-22,1.000432,1.000492,1.001316,1.000028,1.000733,1.000641,1.001079,1.000161,1.000370,0.999780,1.000682,1.000463,1.000296,1.000737,1.000195,1.001845,1.000378,1.000196,1.001480,1.000189,1.000681,1.005184,1.000139,1.000139,1.000139,1.001294,1.000139,1.000196,1.001127,1.001226,1.000139,1.001124,1.001687,1.000809,1.002113,1.000228,1.000123,1.001342,1.000299,1.000339,...,1.003301,1.000139,1.002598,1.000373,1.001691,1.000299,1.000139,1.004797,1.000123,1.000139,1.000125,1.000123,1.001186,1.001047,1.001235,1.001700,1.000258,1.001088,1.000327,1.001429,1.000553,1.000123,1.001315,1.000141,1.001885,1.000768,1.000139,1.000139,1.000321,1.000139,1.000203,1.003410,1.000254,1.001611,1.000139,1.000641,1.001224,1.000297,1.000123,1.000139
2022-07-25,1.001684,1.000161,1.000861,1.000146,1.002988,1.000338,1.000614,1.000139,1.002121,0.999889,1.000503,1.001419,1.000299,1.000287,1.000281,1.000298,1.000248,1.000161,1.001506,1.000518,1.000907,1.004817,1.000123,1.000123,1.000281,1.000098,1.000139,1.000082,1.000139,1.001641,1.000000,1.000844,1.000211,1.000114,0.999776,1.000228,1.000139,1.000806,1.000142,1.000558,...,1.002120,1.000045,1.005079,1.000331,1.002996,1.000304,1.000139,1.005053,1.000123,1.000939,1.001125,1.000185,1.000482,1.000123,1.003107,1.000755,1.001028,1.001690,1.003025,1.000350,1.002188,1.000123,1.001186,1.002006,1.001604,1.000187,1.000123,1.000102,1.000229,1.000139,1.000161,1.003914,1.000159,1.001884,1.000053,1.001850,1.001231,1.000273,1.000123,1.000000
2022-07-26,1.000281,1.000161,1.000148,1.000161,1.003561,1.000340,1.000854,1.000139,1.006418,1.000139,1.000139,1.001622,1.000296,1.000156,1.000296,1.001860,1.001383,1.000139,1.000057,1.004403,1.000165,1.001694,1.000139,1.000139,1.000296,1.000148,1.000139,1.000000,1.000246,1.000161,1.000139,0.999881,1.000302,1.000241,0.999706,1.000195,1.000123,1.000139,1.000299,1.000789,...,1.000385,1.000161,1.001362,1.000159,1.000381,1.000159,1.000159,1.005828,1.000139,1.000139,1.000098,1.000022,1.000264,1.003488,1.002789,1.001092,1.000296,1.000365,1.001983,1.000022,1.000321,1.000139,1.000321,1.001504,1.000308,1.000139,1.000139,1.000176,1.000159,1.000000,1.000358,1.001743,1.001761,1.000292,1.000139,1.004947,1.000139,1.000320,1.000139,1.000202
2022-07-27,1.000165,1.000161,1.000418,1.000161,1.001506,1.000117,1.000139,1.000139,1.001977,1.000139,1.000175,1.002358,1.000279,1.000563,1.000264,1.002297,1.000339,1.000048,1.000360,0.999987,1.000161,1.000728,1.000123,1.000139,1.000279,1.000642,1.000139,1.000165,1.001244,1.000120,1.000123,1.000127,1.000338,1.000443,1.005384,1.000147,1.000139,1.001140,1.000282,1.002121,...,1.000445,1.000145,1.000584,1.001685,1.003929,1.000304,1.000078,1.003937,1.000123,1.000161,1.000165,1.000182,1.000250,1.000173,1.000051,1.000161,1.001285,1.001010,1.001665,1.000277,1.000434,1.000139,1.001160,1.000780,1.001265,1.000145,1.000139,1.000128,1.001817,1.000094,1.000054,1.000381,1.001002,1.001905,1.000123,1.024626,1.000165,1.000577,1.000123,1.000123
2022-07-28,1.000123,1.000161,1.001093,1.000274,1.001758,1.000979,1.000219,1.000072,1.002465,1.0

In [44]:
# 保存预测结果
# 在回测中可以任意选topK买入
pre_df.to_csv('predict.csv')
tar_df.to_csv('target.csv')

### 第一种选股方法
根据预测值排序选股

In [45]:
dff = pd.read_csv("predict.csv",index_col=0)
dff2 = pd.read_csv("target.csv",index_col=0)

In [46]:
dff.head()

,601088.XSHG,000895.XSHE,603986.XSHG,002916.XSHE,600276.XSHG,002812.XSHE,688126.XSHG,601186.XSHG,000408.XSHE,000063.XSHE,600031.XSHG,601236.XSHG,600028.XSHG,600309.XSHG,600015.XSHG,300015.XSHE,600009.XSHG,600436.XSHG,300433.XSHE,600383.XSHG,600111.XSHG,300782.XSHE,000166.XSHE,601939.XSHG,601328.XSHG,688111.XSHG,600999.XSHG,000001.XSHE,300142.XSHE,300750.XSHE,601398.XSHG,688008.XSHG,600132.XSHG,002230.XSHE,688363.XSHG,601169.XSHG,600332.XSHG,600763.XSHG,601988.XSHG,600600.XSHG,...,601877.XSHG,002594.XSHE,600011.XSHG,603290.XSHG,002821.XSHE,003816.XSHE,601009.XSHG,688005.XSHG,600061.XSHG,000800.XSHE,603806.XSHG,002064.XSHE,002493.XSHE,603019.XSHG,002466.XSHE,002074.XSHE,002352.XSHE,002648.XSHE,603185.XSHG,600585.XSHG,600406.XSHG,600958.XSHG,000938.XSHE,300496.XSHE,002938.XSHE,601788.XSHG,601881.XSHG,601899.XSHG,603259.XSHG,601336.XSHG,000002.XSHE,002709.XSHE,000596.XSHE,300122.XSHE,600887.XSHG,002050.XSHE,002414.XSHE,300896.XSHE,601857.XSHG,601658.XSHG
2022-07-22,1.000432,1.000492,1.001316,1.000028,1.000733,1.000641,1.001079,1.000161,1.000370,0.999780,1.000682,1.000463,1.000296,1.000737,1.000195,1.001845,1.000378,1.000196,1.001480,1.000189,1.000681,1.005184,1.000139,1.000139,1.000139,1.001294,1.000139,1.000196,1.001127,1.001226,1.000139,1.001124,1.001687,1.000809,1.002113,1.000228,1.000123,1.001342,1.000299,1.000339,...,1.003301,1.000139,1.002598,1.000373,1.001691,1.000299,1.000139,1.004797,1.000123,1.000139,1.000125,1.000123,1.001186,1.001047,1.001235,1.001700,1.000258,1.001088,1.000327,1.001429,1.000553,1.000123,1.001315,1.000141,1.001885,1.000768,1.000139,1.000139,1.000321,1.000139,1.000203,1.003410,1.000254,1.001611,1.000139,1.000641,1.001224,1.000297,1.000123,1.000139
2022-07-25,1.001684,1.000161,1.000861,1.000146,1.002988,1.000338,1.000614,1.000139,1.002121,0.999889,1.000503,1.001419,1.000299,1.000287,1.000281,1.000298,1.000248,1.000161,1.001506,1.000518,1.000907,1.004817,1.000123,1.000123,1.000281,1.000098,1.000139,1.000082,1.000139,1.001641,1.000000,1.000844,1.000211,1.000114,0.999776,1.000228,1.000139,1.000806,1.000142,1.000558,...,1.002120,1.000045,1.005079,1.000331,1.002996,1.000304,1.000139,1.005053,1.000123,1.000939,1.001125,1.000185,1.000482,1.000123,1.003107,1.000755,1.001028,1.001690,1.003025,1.000350,1.002188,1.000123,1.001186,1.002006,1.001604,1.000187,1.000123,1.000102,1.000229,1.000139,1.000161,1.003914,1.000159,1.001884,1.000053,1.001850,1.001231,1.000273,1.000123,1.000000
2022-07-26,1.000281,1.000161,1.000148,1.000161,1.003561,1.000340,1.000854,1.000139,1.006418,1.000139,1.000139,1.001622,1.000296,1.000156,1.000296,1.001860,1.001383,1.000139,1.000057,1.004403,1.000165,1.001694,1.000139,1.000139,1.000296,1.000148,1.000139,1.000000,1.000246,1.000161,1.000139,0.999881,1.000302,1.000241,0.999706,1.000195,1.000123,1.000139,1.000299,1.000789,...,1.000385,1.000161,1.001362,1.000159,1.000381,1.000159,1.000159,1.005828,1.000139,1.000139,1.000098,1.000022,1.000264,1.003488,1.002789,1.001092,1.000296,1.000365,1.001983,1.000022,1.000321,1.000139,1.000321,1.001504,1.000308,1.000139,1.000139,1.000176,1.000159,1.000000,1.000358,1.001743,1.001761,1.000292,1.000139,1.004947,1.000139,1.000320,1.000139,1.000202
2022-07-27,1.000165,1.000161,1.000418,1.000161,1.001506,1.000117,1.000139,1.000139,1.001977,1.000139,1.000175,1.002358,1.000279,1.000563,1.000264,1.002297,1.000339,1.000048,1.000360,0.999987,1.000161,1.000728,1.000123,1.000139,1.000279,1.000642,1.000139,1.000165,1.001244,1.000120,1.000123,1.000127,1.000338,1.000443,1.005384,1.000147,1.000139,1.001140,1.000282,1.002121,...,1.000445,1.000145,1.000584,1.001685,1.003929,1.000304,1.000078,1.003937,1.000123,1.000161,1.000165,1.000182,1.000250,1.000173,1.000051,1.000161,1.001285,1.001010,1.001665,1.000277,1.000434,1.000139,1.001160,1.000780,1.001265,1.000145,1.000139,1.000128,1.001817,1.000094,1.000054,1.000381,1.001002,1.001905,1.000123,1.024626,1.000165,1.000577,1.000123,1.000123
2022-07-28,1.000123,1.000161,1.001093,1.000274,1.001758,1.000979,1.000219,1.000072,1.002465,1.0

In [47]:
indexlist = dff['2023':].index.tolist()
dict_position = {}

for x in indexlist:
    # 排序选股
    temp = dff.loc[x].sort_values(ascending=False).head(3).index.tolist()
    
    dict_position[x] = temp 


In [48]:
dict_position

{'2023-01-03': ['600050.XSHG', '002252.XSHE', '300413.XSHE'],
 '2023-01-04': ['000938.XSHE', '601155.XSHG', '300769.XSHE'],
 '2023-01-05': ['002241.XSHE', '002920.XSHE', '300896.XSHE'],
 '2023-01-06': ['603806.XSHG', '000301.XSHE', '601877.XSHG'],
 '2023-01-09': ['601877.XSHG', '601899.XSHG', '002304.XSHE'],
 '2023-01-10': ['000425.XSHE', '600085.XSHG', '688599.XSHG'],
 '2023-01-11': ['002252.XSHE', '688599.XSHG', '000938.XSHE'],
 '2023-01-12': ['688005.XSHG', '002493.XSHE', '300919.XSHE'],
 '2023-01-13': ['603486.XSHG', '002311.XSHE', '601066.XSHG'],
 '2023-01-16': ['002821.XSHE', '600276.XSHG', '603486.XSHG'],
 '2023-01-17': ['605117.XSHG', '300769.XSHE', '300122.XSHE'],
 '2023-01-18': ['002555.XSHE', '002821.XSHE', '300759.XSHE'],
 '2023-01-19': ['300759.XSHE', '002230.XSHE', '002180.XSHE'],
 '2023-01-20': ['600050.XSHG', '603369.XSHG', '600845.XSHG'],
 '2023-01-30': ['601689.XSHG', '002230.XSHE', '300896.XSHE'],
 '2023-01-31': ['002120.XSHE', '300759.XSHE', '688008.XSHG'],
 '2023-0

### 第二种选股方法

根据模型在近十日中个股的误差和预测值进行排序

_这个方法可能效果不好_

In [49]:
mae_df = dff-dff2
mae_df = mae_df.abs().rolling(10).mean()
mae_df.head()

,000001.XSHE,000002.XSHE,000063.XSHE,000069.XSHE,000100.XSHE,000157.XSHE,000166.XSHE,000301.XSHE,000333.XSHE,000338.XSHE,000408.XSHE,000425.XSHE,000538.XSHE,000568.XSHE,000596.XSHE,000625.XSHE,000651.XSHE,000661.XSHE,000708.XSHE,000723.XSHE,000725.XSHE,000733.XSHE,000768.XSHE,000776.XSHE,000786.XSHE,000792.XSHE,000800.XSHE,000858.XSHE,000876.XSHE,000877.XSHE,000895.XSHE,000938.XSHE,000963.XSHE,000977.XSHE,001289.XSHE,001979.XSHE,002001.XSHE,002007.XSHE,002027.XSHE,002049.XSHE,...,601985.XSHG,601988.XSHG,601989.XSHG,601995.XSHG,601998.XSHG,603019.XSHG,603185.XSHG,603195.XSHG,603259.XSHG,603260.XSHG,603288.XSHG,603290.XSHG,603369.XSHG,603392.XSHG,603486.XSHG,603501.XSHG,603659.XSHG,603799.XSHG,603806.XSHG,603833.XSHG,603899.XSHG,603986.XSHG,603993.XSHG,605117.XSHG,605499.XSHG,688005.XSHG,688008.XSHG,688012.XSHG,688036.XSHG,688065.XSHG,688111.XSHG,688126.XSHG,688187.XSHG,688303.XSHG,688363.XSHG,688396.XSHG,688561.XSHG,688599.XSHG,688981.XSHG,target2
2022-07-22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-07-25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-07-26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-07-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-07-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [50]:
# 因为我们的标签使用的是2日后的收盘价，所以评估效果的数据进行下移两个位置的操作 防止使用未来信息
mae_df = mae_df.shift(2)['2023':]
mae_df.head()

,000001.XSHE,000002.XSHE,000063.XSHE,000069.XSHE,000100.XSHE,000157.XSHE,000166.XSHE,000301.XSHE,000333.XSHE,000338.XSHE,000408.XSHE,000425.XSHE,000538.XSHE,000568.XSHE,000596.XSHE,000625.XSHE,000651.XSHE,000661.XSHE,000708.XSHE,000723.XSHE,000725.XSHE,000733.XSHE,000768.XSHE,000776.XSHE,000786.XSHE,000792.XSHE,000800.XSHE,000858.XSHE,000876.XSHE,000877.XSHE,000895.XSHE,000938.XSHE,000963.XSHE,000977.XSHE,001289.XSHE,001979.XSHE,002001.XSHE,002007.XSHE,002027.XSHE,002049.XSHE,...,601985.XSHG,601988.XSHG,601989.XSHG,601995.XSHG,601998.XSHG,603019.XSHG,603185.XSHG,603195.XSHG,603259.XSHG,603260.XSHG,603288.XSHG,603290.XSHG,603369.XSHG,603392.XSHG,603486.XSHG,603501.XSHG,603659.XSHG,603799.XSHG,603806.XSHG,603833.XSHG,603899.XSHG,603986.XSHG,603993.XSHG,605117.XSHG,605499.XSHG,688005.XSHG,688008.XSHG,688012.XSHG,688036.XSHG,688065.XSHG,688111.XSHG,688126.XSHG,688187.XSHG,688303.XSHG,688363.XSHG,688396.XSHG,688561.XSHG,688599.XSHG,688981.XSHG,target2
2023-01-03,0.019217,0.017068,0.015952,0.016184,0.015167,0.014848,0.007566,0.033962,0.016785,0.013889,0.025271,0.015907,0.014747,0.023233,0.012275,0.025658,0.013440,0.017298,0.011074,0.018838,0.014747,0.025699,0.027145,0.011828,0.018214,0.017549,0.019519,0.019480,0.015872,0.017540,0.013710,0.019358,0.033217,0.021890,0.028050,NaN,0.018900,0.039787,0.017084,0.026752,...,0.010406,0.006644,0.007752,0.011865,0.013070,0.017308,0.028034,0.018115,0.023043,0.023199,0.015726,0.022540,0.018629,0.016858,0.022974,0.018361,0.026949,0.022201,0.036650,0.018005,0.023047,0.022670,0.017696,0.027739,0.022288,0.027224,0.013242,0.027380,0.027969,0.024061,0.034778,0.025947,0.015778,0.027655,0.035415,0.017087,0.039454,0.037260,0.013304,NaN
2023-01-04,0.023614,0.018097,0.014525,0.015350,0.016728,0.012823,0.004874,0.032763,0.015955,0.013135,0.024262,0.016237,0.010146,0.018431,0.010572,0.024036,0.013573,0.019612,0.008849,0.017102,0.014400,0.026666,0.026035,0.011089,0.018243,0.015177,0.019795,0.015151,0.012666,0.015380,0.011512,0.030270,0.028887,0.025127,0.029864,NaN,0.012622,0.031984,0.016656,0.027507,...,0.009267,0.006284,0.006912,0.008777,0.012042,0.018589,0.027863,0.016693,0.020206,0.023638,0.015462,0.023267,0.014718,0.013348,0.019261,0.017635,0.028129,0.019444,0.037287,0.016034,0.021593,0.022403,0.014552,0.031156,0.021917,0.024362,0.013419,0.024112,0.027354,0.022401,0.035146,0.021470,0.015251,0.026611,0.034579,0.013783,0.040621,0.038682,0.012445,NaN
2023-01-05,0.027077,0.020340,0.013616,0.020006,0.016358,0.012864,0.004749,0.028695,0.017968,0.015727,0.024181,0.017671,0.009778,0.020798,0.017953,0.020273,0.017417,0.027204,0.011213,0.015956,0.014585,0.026979,0.024742,0.013999,0.025617,0.012499,0.017611,0.017661,0.011483,0.017131,0.011005,0.048862,0.029975,0.023460,0.028530,0.033346,0.011775,0.029911,0.023322,0.027247,...,0.009478,0.006528,0.006274,0.008269,0.012034,0.017495,0.026096,0.014935,0.025070,0.023025,0.016019,0.022498,0.020443,0.017599,0.022450,0.019139,0.024726,0.019923,0.036033,0.012010,0.021772,0.022527,0.013518,0.030997,0.022667,0.021460,0.016884,0.022632,0.026570,0.020744,0.035781,0.017884,0.016665,0.024938,0.035004,0.011931,0.038537,0.037349,0.011853,NaN
2023-01-06,0.027883,0.020718,0.013479,0.022948,0.012607,0.011561,0.004466,0.035256,0.017588,0.017710,0.021609,0.016404,0.009120,0.023303,0.028079,0.021669,0.017812,0.033632,0.014428,0.013560,0.016228,0.026007,0.021102,0.016726,0.023784,0.012143,0.013685,0.020591,0.009316,0.017167,0.008819,0.050044,0.027641,0.022587,0.028581,0.029174,0.013701,0.031272,0.021052,0.029650,...,0.008157,0.006495,0.005123,0.009376,0.012553,0.016406,0.025797,0.015319,0.028934,0.026889,0.018715,0.023021,0.024103,0.022382,0.024263,0.021502,0.026854,0.025532,0.046107,0.011719,0.016422,0.023874,0.012604,0.037948,0.020531,0.023289,0.016607,0.019168,0.022035,0.017646,0.036715,0.014989,0.017902,0.020418,0.030156,0.011216,0.040370,0.045095,0.010553,NaN
2023-01-09,0.029384,0.023226,0.016515,0.024689,0.012499,0.010992,0.004685,0.033644,0.017801,0.017819,0.020997,0.015713,0.008485,0

In [51]:
indexlist = dff['2023':].index.tolist()
dict_position2 = {}

for x in indexlist:
    # 排序选股
    temp = pd.concat([dff.loc[x],-mae_df.loc[x]],axis=1,keys=['s1', 's2']).dropna()
    
    dict_position2[x] = temp.sort_values(by=['s1', 's2'],ascending=False).head(3).index.tolist()
    
    
dict_position2

/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:6: FutureWarning: Sorting because non-concatenation axis is not aligned. A future version
of pandas will change to not sort by default.

To accept the future behavior, pass 'sort=False'.

To retain the current behavior and silence the warning, pass 'sort=True'.

  


{'2023-01-03': ['600050.XSHG', '002252.XSHE', '300413.XSHE'],
 '2023-01-04': ['000938.XSHE', '601155.XSHG', '300769.XSHE'],
 '2023-01-05': ['002241.XSHE', '002920.XSHE', '300896.XSHE'],
 '2023-01-06': ['603806.XSHG', '000301.XSHE', '601877.XSHG'],
 '2023-01-09': ['601877.XSHG', '601899.XSHG', '002304.XSHE'],
 '2023-01-10': ['000425.XSHE', '600085.XSHG', '688599.XSHG'],
 '2023-01-11': ['002252.XSHE', '688599.XSHG', '603806.XSHG'],
 '2023-01-12': ['688005.XSHG', '002493.XSHE', '300919.XSHE'],
 '2023-01-13': ['603486.XSHG', '002311.XSHE', '601066.XSHG'],
 '2023-01-16': ['002821.XSHE', '600276.XSHG', '603486.XSHG'],
 '2023-01-17': ['605117.XSHG', '300769.XSHE', '300122.XSHE'],
 '2023-01-18': ['002555.XSHE', '002821.XSHE', '300759.XSHE'],
 '2023-01-19': ['300759.XSHE', '002230.XSHE', '002180.XSHE'],
 '2023-01-20': ['600050.XSHG', '603369.XSHG', '603486.XSHG'],
 '2023-01-30': ['601689.XSHG', '002230.XSHE', '002812.XSHE'],
 '2023-01-31': ['002120.XSHE', '300759.XSHE', '688008.XSHG'],
 '2023-0